In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SmartHomeEnergyETL") \
    .getOrCreate()

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

data = [
    (101,"Living Room","2026-06-01 08:00:00",2.5),
    (101,"Living Room","2026-06-01 19:00:00",3.2),
    (102,"Kitchen","2026-06-02 10:00:00",4.0),
    (102,"Kitchen","2026-06-02 20:00:00",5.1),
    (103,"Bedroom","2026-06-03 14:00:00",1.8),
    (103,"Bedroom","2026-06-03 21:00:00",2.7),
    (104,"Office","2026-06-04 09:00:00",6.5),
    (104,"Office","2026-06-04 18:30:00",7.2),
    (105,"Garage","2026-06-05 11:00:00",3.6),
    (105,"Garage","2026-06-05 22:00:00",4.4)
]

columns = [
    "device_id",
    "room",
    "timestamp",
    "energy_kwh"
]

df = spark.createDataFrame(data, columns)

df = df.withColumn("timestamp", to_timestamp("timestamp"))

df.show()

+---------+-----------+-------------------+----------+
|device_id|       room|          timestamp|energy_kwh|
+---------+-----------+-------------------+----------+
|      101|Living Room|2026-06-01 08:00:00|       2.5|
|      101|Living Room|2026-06-01 19:00:00|       3.2|
|      102|    Kitchen|2026-06-02 10:00:00|       4.0|
|      102|    Kitchen|2026-06-02 20:00:00|       5.1|
|      103|    Bedroom|2026-06-03 14:00:00|       1.8|
|      103|    Bedroom|2026-06-03 21:00:00|       2.7|
|      104|     Office|2026-06-04 09:00:00|       6.5|
|      104|     Office|2026-06-04 18:30:00|       7.2|
|      105|     Garage|2026-06-05 11:00:00|       3.6|
|      105|     Garage|2026-06-05 22:00:00|       4.4|
+---------+-----------+-------------------+----------+



In [0]:
from pyspark.sql.functions import *

df = df.withColumn("date", to_date("timestamp"))

df = df.withColumn("week", weekofyear("timestamp"))

df.show()

+---------+-----------+-------------------+----------+----------+----+
|device_id|       room|          timestamp|energy_kwh|      date|week|
+---------+-----------+-------------------+----------+----------+----+
|      101|Living Room|2026-06-01 08:00:00|       2.5|2026-06-01|  23|
|      101|Living Room|2026-06-01 19:00:00|       3.2|2026-06-01|  23|
|      102|    Kitchen|2026-06-02 10:00:00|       4.0|2026-06-02|  23|
|      102|    Kitchen|2026-06-02 20:00:00|       5.1|2026-06-02|  23|
|      103|    Bedroom|2026-06-03 14:00:00|       1.8|2026-06-03|  23|
|      103|    Bedroom|2026-06-03 21:00:00|       2.7|2026-06-03|  23|
|      104|     Office|2026-06-04 09:00:00|       6.5|2026-06-04|  23|
|      104|     Office|2026-06-04 18:30:00|       7.2|2026-06-04|  23|
|      105|     Garage|2026-06-05 11:00:00|       3.6|2026-06-05|  23|
|      105|     Garage|2026-06-05 22:00:00|       4.4|2026-06-05|  23|
+---------+-----------+-------------------+----------+----------+----+



In [0]:
daily_summary = df.groupBy(
    "date",
    "room"
).agg(
    sum("energy_kwh").alias("daily_energy")
)

daily_summary.show()

+----------+-----------+------------+
|      date|       room|daily_energy|
+----------+-----------+------------+
|2026-06-01|Living Room|         5.7|
|2026-06-02|    Kitchen|         9.1|
|2026-06-03|    Bedroom|         4.5|
|2026-06-04|     Office|        13.7|
|2026-06-05|     Garage|         8.0|
+----------+-----------+------------+



In [0]:
weekly_summary = df.groupBy(
    "week",
    "room"
).agg(
    sum("energy_kwh").alias("weekly_energy")
)

weekly_summary.show()

+----+-----------+-------------+
|week|       room|weekly_energy|
+----+-----------+-------------+
|  23|Living Room|          5.7|
|  23|    Kitchen|          9.1|
|  23|    Bedroom|          4.5|
|  23|     Office|         13.7|
|  23|     Garage|          8.0|
+----+-----------+-------------+



In [0]:
daily_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/tmp/daily_energy_delta")

weekly_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/tmp/weekly_energy_delta")

In [0]:
daily_summary.write \
    .option("header",True) \
    .mode("overwrite") \
    .csv("/tmp/daily_energy_csv")

weekly_summary.write \
    .option("header",True) \
    .mode("overwrite") \
    .csv("/tmp/weekly_energy_csv")

In [0]:
df.createOrReplaceTempView("energy_logs")

In [0]:
%sql
SELECT
    room,
    SUM(energy_kwh) AS total_energy
FROM energy_logs
GROUP BY room
ORDER BY total_energy DESC;

room,total_energy
Office,13.7
Kitchen,9.1
Garage,8.0
Living Room,5.7
Bedroom,4.5
